In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
import os

In [3]:
gene_expression_matrix = pd.read_parquet("data/gene_expression_matrix.parquet")

transposed = (
    gene_expression_matrix
    .drop(columns=['gene_name', 'gene_biotype'])
    .set_index('gene_id')
    .T
)
transposed.index.name = 'sample'

# Add categorical sample_type column
transposed.insert(0, 'sample_type', pd.Categorical(transposed.index.str.split('_').str[0]))

# Standardize and run PCA
X = transposed.drop(columns=['Target(SNHG14)', 'sample_type']).values

# Scree plot — inspect to choose N_COMPONENTS in the next cell
n_max = min(len(transposed), 50)
pca_full = PCA(n_components=n_max, random_state=42)
pca_full.fit(StandardScaler().fit_transform(X))

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",48
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD

In [4]:
gene_expression_matrix.head()

,gene_id,gene_name,gene_biotype,ZDS2_1_2_cpm,SP1R_1_cpm,SP1R_2_cpm,nZF36_1_cpm,nZF36_2_cpm,nZF42_1_cpm,nZF42_2_cpm,...,nZF93_1_cpm,nZF93_2_cpm,nZF105_1_cpm,nZF105_2_cpm,Control_2(HSR6),hATF555R_1_cpm,hATF555R_2_cpm,hATF555Q_1_cpm,hATF555Q_2_cpm,ZDS2_1_2_cpm.1
0,ENSG00000000003,TSPAN6,protein_coding,7.084306,6.905406,6.854844,5.948875,7.740376,7.280572,10.008006,...,7.767628,9.530814,8.714803,9.119923,7.037930,8.706457,7.477779,8.111175,7.708378,6.505686
1,ENSG00000000005,TNMD,protein_coding,0.104181,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ENSG00000000419,DPM1,protein_coding,53.270856,54.658290,49.959774,56.318380,53.638417,56.771391,57.996635,...,62.137761,59.594658,58.258044,59.340350,55.548466,53.788898,59.725558,54.953696,55.799528,58.331454
3,ENSG00000000457,SCYL3,protein_coding,5.834134,5.393228,4.909551,4.332183,4.894016,6.426988,3.893767,...,4.567913,3.845927,5.728626,5.849128,5.118495,5.688636,4.411392,5.277150,5.328604,4.349764
4,ENSG00000000460,FIRRM,protein_coding,10.626459,10.060444,9.263303,10.141247,8.221947,9.740903,8.504807,...,9.897146,10.626903,7.603449,8.962374,12.205642,8.627765,9.292082,8.795249,9.450731,9.703320


In [4]:
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
scree_df = pd.DataFrame({'PC': range(1, n_max + 1), 'Cumulative Variance (%)': cumvar})

fig_scree = px.line(
    scree_df, x='PC', y='Cumulative Variance (%)',
    markers=True,
    title='Scree Plot — Cumulative Explained Variance by Number of PCs'
         '<br><sup>Choose N_COMPONENTS in the next cell based on your desired variance threshold</sup>',
)
fig_scree.add_hline(y=80, line_dash='dash', line_color='orange', annotation_text='80%', annotation_position='right')
fig_scree.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='90%', annotation_position='right')
fig_scree.update_layout(xaxis_title='Number of Principal Components', yaxis_title='Cumulative Variance Explained (%)')
fig_scree.show()

n_80 = int(np.searchsorted(cumvar, 80)) + 1
n_90 = int(np.searchsorted(cumvar, 90)) + 1
print(f'PCs needed for 80% variance: {n_80}')
print(f'PCs needed for 90% variance: {n_90}')

PCs needed for 80% variance: 36
PCs needed for 90% variance: 41


In [5]:
# ── Set N after inspecting the scree plot above ──
N_COMPONENTS = 36

pca = PCA(n_components=N_COMPONENTS, random_state=42)
pcs = pca.fit_transform(StandardScaler().fit_transform(X))

pc_cols = [f'PC{i+1}' for i in range(N_COMPONENTS)]
pca_df = pd.DataFrame(pcs, columns=pc_cols, index=transposed.index)
pca_df['sample_type'] = transposed['sample_type'].values

print(f'Using {N_COMPONENTS} PCs — explains {pca.explained_variance_ratio_.sum():.1%} of total variance')

Using 36 PCs — explains 81.2% of total variance


In [6]:
# 2D PCA scatter (PC1 vs PC2) — all treatments, text labels
control_centroid_2d = pca_df[pca_df['sample_type'] == 'Control'][['PC1', 'PC2']].mean()

fig1 = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='sample_type',
    text='sample_type',
    hover_name=pca_df.index,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    },
    title='PCA of Gene Expression by All Sample Types'
          '<br><sup>Treatments closest to Control in 2D are safest visually; use N-D distance below for full picture</sup>',
)
fig1.update_traces(mode='text')
fig1.update_layout(showlegend=False)

for trace in fig1.data:
    if trace.name == 'Control':
        trace.textfont.color = 'black'
        trace.textfont.size = 14
        trace.textfont.weight = 'bold'
    else:
        trace.textfont.color = trace.marker.color
        trace.textfont.size = 10

fig1.add_trace(go.Scatter(
    x=[control_centroid_2d['PC1']],
    y=[control_centroid_2d['PC2']],
    mode='text',
    text=['⊕ Control Mean'],
    textfont=dict(color='black', size=16, weight='bold'),
    hoverinfo='skip',
    showlegend=False,
))
fig1.show()

In [7]:
# Log2 fold change of Target gene per treatment
target_mean = transposed.groupby('sample_type')['Target(SNHG14)'].mean()
control_mean = target_mean['Control']
log2fc = np.log2(target_mean / control_mean).drop('Control').sort_values()

fig2 = px.bar(
    log2fc,
    x=log2fc.index,
    y=log2fc.values,
    labels={'x': 'Treatment', 'y': 'Log2 Fold Change vs Control'},
    title='Target Gene (SNHG14) Log2 Fold Change by Treatment (Effectiveness)'
          '<br><sup>Log2FC: −1 = halved (50% reduction), −2 = quartered (75% reduction), 0 = no change relative to control</sup>',
    color=log2fc.values,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
)
fig2.add_hline(y=0, line_dash='dash', line_color='black')
fig2.update_layout(coloraxis_showscale=False, xaxis_tickangle=-45)

tick_log2fc = np.array([-1.2, -1.0, -0.8, -0.6, -0.4, -0.2, 0.0])
tick_pct = (2 ** tick_log2fc - 1) * 100
y_min, y_max = log2fc.min() * 1.15, log2fc.max() * 1.15

fig2.add_trace(go.Scatter(x=[None], y=[None], yaxis='y2', showlegend=False))
fig2.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    yaxis2=dict(
        overlaying='y', side='right', range=[y_min, y_max],
        tickmode='array', tickvals=tick_log2fc,
        ticktext=[f'{p:.0f}%' for p in tick_pct],
        title='% Change vs Control',
    ),
)
fig2.show()

In [8]:
# N-dimensional Euclidean distance to Control centroid
control_centroid_nd = pca_df[pca_df['sample_type'] == 'Control'][pc_cols].mean()

pca_df['dist_to_control'] = np.sqrt(
    ((pca_df[pc_cols] - control_centroid_nd) ** 2).sum(axis=1)
)

dist_summary = (
    pca_df[pca_df['sample_type'] != 'Control']
    .groupby('sample_type')['dist_to_control']
    .agg(mean_dist='mean', std_dist='std', n='count')
    .sort_values('mean_dist')
)
dist_summary['log2FC'] = log2fc.reindex(dist_summary.index)

fig3 = px.bar(
    dist_summary,
    x=dist_summary.index,
    y='mean_dist',
    error_y='std_dist',
    color='log2FC',
    color_continuous_scale='Reds_r',
    range_color=[log2fc.min(), 0],
    labels={'x': 'Treatment', 'mean_dist': f'Mean Distance to Control Centroid ({N_COMPONENTS} PCs)', 'log2FC': 'Log2FC'},
    title=f'Consistency with Control — Euclidean Distance in {N_COMPONENTS}-PC Space'
          '<br><sup>Bar height = distance to Control (lower = safer); color = log2FC effectiveness (darker red = more effective)</sup>',
)
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

dist_summary

,mean_dist,std_dist,n,log2FC
sample_type,,,,
hATF567,220.333767,5.805069,2,-0.501195
nZF42,225.616336,53.069655,2,-0.079278
nZF154,226.035046,7.784296,2,-0.313456
hATF561,227.240572,18.787523,2,-0.567325
SP1R,227.852318,47.122121,2,0.078562
nZF93,231.661933,31.732467,2,-0.415652
nZF145,232.156937,58.030604,2,-0.020723
Base,236.627428,28.189161,2,-0.119212
nZF36,257.274753,7.445047,2,-0.015802


In [9]:
# Effectiveness vs Safety tradeoff scatter with Pareto frontier

sorted_df = dist_summary.sort_values('log2FC')
pareto, min_dist = [], float('inf')
for name, row in sorted_df.iterrows():
    if row['mean_dist'] < min_dist:
        pareto.append((row['log2FC'], row['mean_dist']))
        min_dist = row['mean_dist']
pareto_x = [p[0] for p in pareto]
pareto_y = [p[1] for p in pareto]

step_x, step_y = [], []
for i, (fx, fy) in enumerate(zip(pareto_x, pareto_y)):
    if i == 0:
        step_x += [dist_summary['log2FC'].min() * 1.05, fx]
        step_y += [fy, fy]
    else:
        step_x += [pareto_x[i - 1], fx]
        step_y += [fy, fy]
step_x.append(fx)
step_y.append(0)

med_log2fc = dist_summary['log2FC'].median()
med_dist = dist_summary['mean_dist'].median()

fig4 = px.scatter(
    dist_summary,
    x='log2FC', y='mean_dist',
    size='std_dist', size_max=25,
    text=dist_summary.index,
    color=dist_summary.index,
    color_discrete_sequence=px.colors.qualitative.Dark24,
    labels={
        'log2FC': 'Log2 Fold Change (lower = more effective)',
        'mean_dist': f'Mean Distance to Control (lower = safer)',
    },
    title=f'Effectiveness vs Safety Tradeoff ({N_COMPONENTS}-PC Distance)'
          '<br><sup>Bubble size = replicate std dev (larger = less consistent); Pareto frontier = optimal tradeoff candidates</sup>',
)
fig4.update_traces(mode='markers+text', textposition='middle center', textfont=dict(color='black', size=11))

fig4.add_trace(go.Scatter(
    x=step_x, y=step_y, mode='lines',
    line=dict(color='black', width=2, dash='dot'),
    name='Pareto Frontier', showlegend=True,
))

fig4.add_vline(x=med_log2fc, line_dash='dash', line_color='gray', opacity=0.5)
fig4.add_hline(y=med_dist, line_dash='dash', line_color='gray', opacity=0.5)

for x, y, text, color, xanchor, yanchor in [
    (0.01, 0.01, '✓ Safe & Effective',  'green',     'left',  'bottom'),
    (0.99, 0.01, 'Safe but Weak',        'steelblue', 'right', 'bottom'),
    (0.01, 0.99, 'Effective but Risky',  'orange',    'left',  'top'),
    (0.99, 0.99, '✗ Unsafe & Weak',      'red',       'right', 'top'),
]:
    fig4.add_annotation(
        x=x, y=y, text=text,
        xref='paper', yref='paper',
        showarrow=False,
        font=dict(color=color, size=11),
        xanchor=xanchor, yanchor=yanchor,
    )

fig4.update_layout(showlegend=False)
fig4.show()

In [10]:
# Final summary table
pareto_names, min_dist_seen = set(), float('inf')
for name, row in dist_summary.sort_values('log2FC').iterrows():
    if row['mean_dist'] < min_dist_seen:
        pareto_names.add(name)
        min_dist_seen = row['mean_dist']

summary = dist_summary[['mean_dist', 'std_dist', 'n', 'log2FC']].copy()
summary['pareto_optimal'] = summary.index.isin(pareto_names)
summary.sort_values(['pareto_optimal', 'log2FC'], ascending=[False, True])
summary.to_csv('data/pca_summary.csv')
summary

,mean_dist,std_dist,n,log2FC,pareto_optimal
sample_type,,,,,
hATF567,220.333767,5.805069,2,-0.501195,True
nZF42,225.616336,53.069655,2,-0.079278,False
nZF154,226.035046,7.784296,2,-0.313456,False
hATF561,227.240572,18.787523,2,-0.567325,True
SP1R,227.852318,47.122121,2,0.078562,False
nZF93,231.661933,31.732467,2,-0.415652,False
nZF145,232.156937,58.030604,2,-0.020723,False
Base,236.627428,28.189161,2,-0.119212,False
nZF36,257.274753,7.445047,2,-0.015802,False


In [11]:
os.makedirs('figures', exist_ok=True)

fig_scree.write_image('figures/01_scree_plot.png')
fig1.write_image('figures/02_pca_scatter_2d.png')
fig2.write_image('figures/03_log2fc_by_treatment.png')
fig3.write_image('figures/04_nd_distance_to_control.png')
fig4.write_image('figures/05_effectiveness_vs_safety_pareto_tradeoff.png')